# 04. Feature Engineering
##### DSR Berlin | Data Science Fundamentals
##### Inspired by Adam Green (adgefficiency.com), Rachel Berryman, and Samson Afolabi

---

## 🎯 Learning Objectives
By the end of this notebook, you will be able to:
1. Explain the philosophy of **Feature Engineering**: transforming raw data into informative signals that linear and non-linear algorithms can exploit.
2. Apply **continuous feature transformations**: log transforms (`np.log1p`) and quantile-based binning (`pd.cut`, `pd.qcut`).
3. Master all core **categorical encoding techniques**:
   - One-Hot Encoding (`OneHotEncoder`)
   - Ordinal & Label Encoding
   - Frequency / Count Encoding
   - Out-of-fold Target / Mean Encoding (and how to avoid catastrophic target leakage)
4. Engineer **domain-specific interaction features** (ratios, bundled service counts).
5. Apply **feature scaling** (`StandardScaler`, `RobustScaler`) strictly within the leak-free train/test discipline.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    RobustScaler,
)
from sklearn.model_selection import KFold

import warnings

warnings.filterwarnings("ignore")

# Load the cleaned partitions prepared in Notebook 03
X_train = pd.read_csv("../data/telco_X_train_clean.csv")
X_test = pd.read_csv("../data/telco_X_test_clean.csv")
y_train = pd.read_csv("../data/telco_y_train.csv")["Churn"]
y_test = pd.read_csv("../data/telco_y_test.csv")["Churn"]

# Map target to binary integer (1 = Churn, 0 = Retained)
y_train_num = (y_train == "Yes").astype(int)
y_test_num = (y_test == "Yes").astype(int)

print(f"Loaded Cleaned Train: {X_train.shape}, Test: {X_test.shape}")
X_train.head(3)

Loaded Cleaned Train: (5639, 20), Test: (1413, 20)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,3594-BDSOA,Female,0,Yes,No,24,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One Year,No,Credit Card (Automatic),20.40,482.80
1,6371-NZYEG,Male,0,Yes,Yes,16,Yes,No,Dsl,Yes,No,Yes,No,Yes,No,Two Year,No,Mailed Check,64.25,1024.00
2,6418-HNFED,Male,0,Yes,No,51,Yes,No,Dsl,Yes,Yes,Yes,Yes,Yes,Yes,Two Year,Yes,Credit Card (Automatic),83.25,4089.45


## 1. Continuous Feature Transformations

Raw continuous features often have heavily right-skewed distributions or non-linear step thresholds.

### A. Log Transformation
When a positive continuous feature has high skew (e.g. `TotalCharges`), taking $\log(1 + x)$ compresses extreme values and brings the distribution closer to normal, which dramatically stabilizes linear and distance-based algorithms.


In [ ]:
# Log transformation on TotalCharges
X_train["TotalCharges_log"] = np.log1p(X_train["TotalCharges"])
X_test["TotalCharges_log"] = np.log1p(X_test["TotalCharges"])

print(f"Original TotalCharges Skew: {X_train['TotalCharges'].skew():.2f}")
print(f"Log TotalCharges Skew:      {X_train['TotalCharges_log'].skew():.2f}")

Original TotalCharges Skew: 0.96
Log TotalCharges Skew:      -0.84


### B. Discretization / Binning (`pd.cut` & `pd.qcut`)
Sometimes business rules or customer behavior change at specific lifecycle milestones.
- **`pd.cut`**: Bins into intervals of equal width or custom domain edges.
- **`pd.qcut`**: Bins into quantiles with an equal number of samples per bin.


In [4]:
# Define domain-specific tenure cohorts (in months):
# [0-12m: New, 12-24m: Onboarding, 24-48m: Established, 48-72m: Loyal]
bins = [-1, 12, 24, 48, 73]
labels = ["New (0-12m)", "Onboarding (1-2y)", "Established (2-4y)", "Loyal (4-6y)"]

X_train["tenure_cohort"] = pd.cut(X_train["tenure"], bins=bins, labels=labels)
X_test["tenure_cohort"] = pd.cut(X_test["tenure"], bins=bins, labels=labels)

print("Tenure cohort distribution in Train:")
print(X_train["tenure_cohort"].value_counts())

Tenure cohort distribution in Train:
tenure_cohort
Loyal (4-6y)          1806
New (0-12m)           1732
Established (2-4y)    1290
Onboarding (1-2y)      811
Name: count, dtype: int64


## 2. Domain-Specific Interaction Features

Senior data scientists consult domain experts to build features that capture customer behavior directly:

1. **Service Stickiness Count**: A customer with 6 bundled add-ons (Security, Backup, Streaming, TechSupport) is significantly less likely to churn than a customer with only raw internet.
2. **Monthly Charge to Tenure Ratio**: Captures financial intensity relative to customer loyalty.


In [ ]:
# 1. Total Add-on Services Count
service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

X_train["total_services"] = (X_train[service_cols] == "Yes").sum(axis=1)
X_test["total_services"] = (X_test[service_cols] == "Yes").sum(axis=1)

# 2. Financial Intensity Ratio
X_train["charge_tenure_ratio"] = X_train["MonthlyCharges"] / (X_train["tenure"] + 1)
X_test["charge_tenure_ratio"] = X_test["MonthlyCharges"] / (X_test["tenure"] + 1)

# 3. Automatic Payment Flag
auto_pay = ["Bank Transfer (Automatic)", "Credit Card (Automatic)"]
X_train["is_auto_payment"] = X_train["PaymentMethod"].isin(auto_pay).astype(int)
X_test["is_auto_payment"] = X_test["PaymentMethod"].isin(auto_pay).astype(int)

print("Engineered domain features summary:")
print(X_train[["total_services", "charge_tenure_ratio", "is_auto_payment"]].describe())

## 3. Categorical Encodings

Machine learning models operate on matrices of numbers. We must convert categorical strings into numerical representations without introducing false ordering.

### A. One-Hot Encoding (Nominal Features)
- For nominal categories with no natural rank (`Contract`, `InternetService`, `PaymentMethod`).
- Creates $K-1$ binary dummy columns (`drop='first'`) to prevent multicollinearity.


In [6]:
nominal_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
ohe.fit(X_train[nominal_cols])

train_ohe_df = pd.DataFrame(
    ohe.transform(X_train[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X_train.index,
)

test_ohe_df = pd.DataFrame(
    ohe.transform(X_test[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X_test.index,
)

print(f"Generated {train_ohe_df.shape[1]} one-hot encoded columns.")
print("Sample columns:", list(train_ohe_df.columns[:5]))

Generated 14 one-hot encoded columns.
Sample columns: ['gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service']


In [8]:
train_ohe_df.columns

Index(['gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber Optic', 'InternetService_No',
       'Contract_One Year', 'Contract_Two Year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit Card (Automatic)',
       'PaymentMethod_Electronic Check', 'PaymentMethod_Mailed Check'],
      dtype='str')

### B. Frequency / Count Encoding
Replaces each category with its relative frequency (percentage of the dataset). Highly effective for categories with moderate-to-high cardinality.


In [ ]:
# Frequency encode PaymentMethod based on Train distribution
payment_freq = X_train["PaymentMethod"].value_counts(normalize=True).to_dict()

X_train["PaymentMethod_freq"] = X_train["PaymentMethod"].map(payment_freq)
X_test["PaymentMethod_freq"] = X_test["PaymentMethod"].map(payment_freq).fillna(0.0)

print("PaymentMethod frequency mapping:")
for k, v in payment_freq.items():
    print(f"  {k:30}: {v*100:.2f}%")

### C. Target / Mean Encoding (with Out-of-Fold Cross-Validation)

Target encoding replaces a category with the average target value ($P(y=1 | \text{Category})$) for that category.

> ⚠️ **THE TARGET LEAKAGE TRAP**:
> If you compute target means on the full training set without cross-validation, a rare category with 1 sample that churned receives an encoded value of $1.00$. The model will overfit completely!
> **Solution**: Use **Out-of-Fold (OOF)** K-Fold encoding during training, and overall training means on the test set!


In [ ]:
store1 = pd.DataFrame(
    {"store": ["A"] * 3, "Sales": [100, 200, 300], "noise": [0, 0, 0]}
)

store2 = pd.DataFrame({"store": ["B"] * 3, "Sales": [10, 20, 30], "noise": [0, 0, 0]})

data = pd.concat([store1, store2], axis=0)
data

In [ ]:
def mean_encode(data, col, on):
    data = data.copy()

    # Compute mean target per category
    means = data.groupby(col)[on].mean()

    # Preserve original column
    data[f"{col}-original"] = data[col]

    # Apply encoding
    data[col] = data[col].map(means)

    # Fill unseen categories with global mean
    data[col] = data[col].fillna(data[on].mean())

    return data


# define data
store1 = pd.DataFrame(
    {"store": ["A"] * 3, "Sales": [100, 200, 300], "noise": [0, 0, 0]}
)

store2 = pd.DataFrame({"store": ["B"] * 3, "Sales": [10, 20, 30], "noise": [0, 0, 0]})

data = pd.concat([store1, store2], axis=0)
data = mean_encode(data, col="store", on="Sales")

data

In [ ]:
def oof_target_encode(train_df, test_df, cat_col, target_series, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    train_encoded = pd.Series(index=train_df.index, dtype=float)
    global_mean = target_series.mean()

    # 1. Out-of-fold computation on Train
    for tr_idx, val_idx in kf.split(train_df):
        tr_slice = train_df.iloc[tr_idx]
        target_slice = target_series.iloc[tr_idx]

        # Calculate conditional target mean
        cat_means = target_slice.groupby(tr_slice[cat_col]).mean()
        train_encoded.iloc[val_idx] = train_df.iloc[val_idx][cat_col].map(cat_means)

    train_encoded = train_encoded.fillna(global_mean)

    # 2. Overall training mean applied to Test
    overall_cat_means = target_series.groupby(train_df[cat_col]).mean()
    test_encoded = test_df[cat_col].map(overall_cat_means).fillna(global_mean)

    return train_encoded, test_encoded


# Apply to Contract
tr_contract_te, te_contract_te = oof_target_encode(
    X_train, X_test, "Contract", y_train_num
)
X_train["Contract_target_enc"] = tr_contract_te
X_test["Contract_target_enc"] = te_contract_te

print("Target Churn Probability by Contract (from Train):")
print(y_train_num.groupby(X_train["Contract"]).mean())

## C. Binary Encoding
Encoding the string using it's binary (0110100 etc) representation

In Python we can do this using the `bin()` builtin:

In [ ]:
# Number 2 in binary as a string
bin(2)

'0b10'

In [9]:
bin(16)

'0b10000'

In [19]:
import category_encoders as ce

data = ["A", "BB", "B", "A", "C", "D", "A", "E"]
# data = [1,2,3,1,4,5,1,6]

enc = (
    ce.BinaryEncoder()
)  # similar to onehot, but stores categories as binary bitstrings.

print("{} classes".format(len(set(data))))

pd.concat([enc.fit_transform(data), pd.DataFrame(data, columns=["data"])], axis=1)

6 classes


,0_0,0_1,0_2,data
0,0,0,1,A
1,0,1,0,BB
2,0,1,1,B
3,0,0,1,A
4,1,0,0,C
5,1,0,1,D
6,0,0,1,A
7,1,1,0,E


In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)

data_df = pd.DataFrame({"data": data})
encoded = ohe.fit_transform(data_df)

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(["data"]))

pd.concat([encoded_df, data_df], axis=1)

,data_A,data_B,data_BB,data_C,data_D,data_E,data
0,1.0,0.0,0.0,0.0,0.0,0.0,A
1,0.0,0.0,1.0,0.0,0.0,0.0,BB
2,0.0,1.0,0.0,0.0,0.0,0.0,B
3,1.0,0.0,0.0,0.0,0.0,0.0,A
4,0.0,0.0,0.0,1.0,0.0,0.0,C
5,0.0,0.0,0.0,0.0,1.0,0.0,D
6,1.0,0.0,0.0,0.0,0.0,0.0,A
7,0.0,0.0,0.0,0.0,0.0,1.0,E


## 4. Feature Scaling (Fit on Train, Transform Test)

Algorithms that rely on geometric distance (K-Nearest Neighbors, SVMs, Logistic Regression, Neural Networks) or gradient descent are heavily distorted if one feature spans $[0, 100]$ and another spans $[0, 10000]$.

- **`StandardScaler`**: $z = \frac{x - \mu}{\sigma}$ (centers at 0, unit variance).
- **`RobustScaler`**: Uses median and IQR, resistant to residual outliers.


In [ ]:
scale_cols = ["tenure", "MonthlyCharges", "TotalCharges_log", "charge_tenure_ratio"]

scaler = StandardScaler()
scaler.fit(X_train[scale_cols])

X_train_scaled = pd.DataFrame(
    scaler.transform(X_train[scale_cols]),
    columns=[c + "_scaled" for c in scale_cols],
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test[scale_cols]),
    columns=[c + "_scaled" for c in scale_cols],
    index=X_test.index,
)

print("Scaled features mean and std on Train:")
print(X_train_scaled.agg(["mean", "std"]).round(3))

## 5. Assembling the Full Engineered Matrix

We now combine our continuous, interaction, frequency, target-encoded, and one-hot features into full modeling matrices.


In [ ]:
# Numeric and engineered features
engineered_numeric = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges_log",
    "total_services",
    "charge_tenure_ratio",
    "is_auto_payment",
    "PaymentMethod_freq",
    "Contract_target_enc",
    "SeniorCitizen",
]

X_train_final = pd.concat([X_train[engineered_numeric], train_ohe_df], axis=1)
X_test_final = pd.concat([X_test[engineered_numeric], test_ohe_df], axis=1)

print(f"Final Engineered Training Matrix: {X_train_final.shape}")
print(f"Final Engineered Test Matrix:     {X_test_final.shape}")

# Save matrices for Feature Selection & Modeling
X_train_final.to_csv("../data/telco_X_train_engineered.csv", index=False)
X_test_final.to_csv("../data/telco_X_test_engineered.csv", index=False)
print("Saved engineered matrices to data/ successfully!")